<a href="https://colab.research.google.com/github/mehanshbarthwal-lab/search-ranking-ml/blob/main/work/notebooks/w05_model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mehanshbarthwal-lab/search-ranking-ml/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Method choice and why

**Methods I'm running, in order:** Logistic Regression (baseline learned model — readable,
gives me weights I can sanity-check), Decision Tree (a second readable model, shallow, so I can
literally trace a path from features to a label), and Random Forest (the flexible model, since
Week 3's honest F1 of 0.135 already came from a forest — I want to see whether that number holds
up, improves, or falls apart once I test it properly against a grouped split and a couple of new
features).

**Why this progression:** per the framework video, you start with the readable model before the
flexible one, and you only reach for something more complex once you've confirmed the simpler
one can't capture the pattern. Logistic Regression tells me if the relationship between my
features and answered_away is close to linear. The Decision Tree tells me if there's a small
number of clean splits that explain most of the pattern. Random Forest tells me whether the
signal is tangled enough that voting across many trees actually helps. I'm not running Gradient
Boosting this week — the video is explicit that boosting should wait until the simpler models
show there's real signal worth chasing, and with only 5-7 features and ~29.5k rows, I don't think
I've earned that complexity yet.

**Features:** I'm keeping Week 3's honest 5-feature frame (`gsc_impressions_prior30`,
`gsc_clicks_prior30`, `gsc_avg_position_prior30`, `word_count`, plus `content_type` and
`main_intent` one-hot encoded) as the base, and adding two new prior-period-only features that
pass the same decision-time test: `gsc_ctr_prior30` (= gsc_clicks_prior30 / gsc_impressions_prior30,
computed only from prior-window columns, so it's not leaking anything from the label window) and
`content_age_days` (static content property, known well before the March decision point, and a
plausible signal since older pages may be more likely to get "answered away" by newer SERP
features). Both are available before the label window closes, so both pass the same leakage test
`click_change_pct` failed in Week 3.

In [18]:
import duckdb
import pandas as pd

con = duckdb.connect()

# HF auth — required fresh in this notebook, Colab scopes secrets per-notebook
try:
    from google.colab import userdata
    hf_token = userdata.get('HF_Token')
except Exception:
    from getpass import getpass
    hf_token = getpass('HF_Token: ')

con.execute(f"CREATE SECRET (TYPE huggingface, TOKEN '{hf_token}')")

FACT = "'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/**/*.parquet'"
DIM = "'hf://datasets/FlyRank/internship-warehouse/dim_content.parquet'"

# Same prior30 (Feb) vs last30 (March) windows as Week 3 — decision date March 31, 2026
query = f"""
WITH prior AS (
    SELECT content_hash_id, client_hash_id,
           SUM(gsc_impressions) AS gsc_impressions_prior30,
           SUM(gsc_clicks) AS gsc_clicks_prior30,
           AVG(gsc_avg_position) AS gsc_avg_position_prior30
    FROM {FACT}
    WHERE month = '2026-02'
    GROUP BY content_hash_id, client_hash_id
),
last AS (
    SELECT content_hash_id, client_hash_id,
           SUM(gsc_impressions) AS gsc_impressions_last30,
           SUM(gsc_clicks) AS gsc_clicks_last30
    FROM {FACT}
    WHERE month = '2026-03'
    GROUP BY content_hash_id, client_hash_id
)
SELECT
    p.content_hash_id, p.client_hash_id,
    p.gsc_impressions_prior30, p.gsc_clicks_prior30, p.gsc_avg_position_prior30,
    l.gsc_impressions_last30, l.gsc_clicks_last30,
    d.word_count, d.content_type, d.main_intent
FROM prior p
JOIN last l USING (content_hash_id, client_hash_id)
JOIN {DIM} d USING (content_hash_id)
WHERE p.gsc_impressions_prior30 >= 50 AND p.gsc_clicks_prior30 >= 3
"""

df = con.sql(query).df()
print(f"Rows: {len(df):,}")

# Same pattern_group logic as Week 3
df['impr_change_pct'] = (df['gsc_impressions_last30'] - df['gsc_impressions_prior30']) / df['gsc_impressions_prior30'] * 100
df['click_change_pct'] = (df['gsc_clicks_last30'] - df['gsc_clicks_prior30']) / df['gsc_clicks_prior30'] * 100

def assign_pattern(row):
    if row['impr_change_pct'] >= -5 and row['click_change_pct'] <= -15:
        return 'answered_away'
    elif row['impr_change_pct'] <= -15 and row['click_change_pct'] <= -15:
        return 'normal_decay'
    else:
        return 'stable_other'

df['pattern_group'] = df.apply(assign_pattern, axis=1)

# New decision-time-safe feature — built only from prior-window columns already in df
df['gsc_ctr_prior30'] = df['gsc_clicks_prior30'] / df['gsc_impressions_prior30']

print(df['pattern_group'].value_counts())
print(f"\nBase rate (answered_away): {(df['pattern_group'] == 'answered_away').mean():.3f}")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Rows: 29,516
pattern_group
stable_other     19186
answered_away     6759
normal_decay      3571
Name: count, dtype: int64

Base rate (answered_away): 0.229


## 2. Split design

**Two splits, run side by side, on the same rows and same features:**

1. **Random split** — `train_test_split(..., stratify=y, random_state=42)`, same as Week 3.
   This is the split that let a page from a client the model already knows show up in both
   train and test. Per the framework video's own forest example on grouped clients, this kind
   of split tends to look artificially strong, since the model can partly recognize *which
   client* a row came from instead of learning the actual `answered_away` pattern.

2. **Grouped split, by `client_hash_id`** — every row from a given client sits entirely on one
   side of the split, never both. This is the honest deployment test, because the real
   decision this model supports is: can it flag risk on a client's pages using patterns learned
   from *other* clients. With one client at roughly a quarter of all rows, a random split can't
   tell me that; only the grouped split can.

I'm running both and putting them in the same table so the gap between them is visible, not
because I want to report the random-split number as real, but because the size of that gap is
itself the finding — it tells me how much of Week 3's 0.135 was actually pattern versus how
much was the model partly recognizing clients.

In [19]:
from sklearn.model_selection import train_test_split, GroupShuffleSplit

y = (df['pattern_group'] == 'answered_away').astype(int)

feature_cols_numeric = [
    'gsc_impressions_prior30', 'gsc_clicks_prior30', 'gsc_avg_position_prior30',
    'word_count', 'gsc_ctr_prior30'
]
feature_cols_categorical = ['content_type', 'main_intent']

X = pd.get_dummies(df[feature_cols_numeric + feature_cols_categorical], dummy_na=True)

# Fill numeric NaNs (e.g. missing word_count) with 0 — categorical NaNs already
# handled by dummy_na=True above, which gives them their own "_nan" indicator column
X[feature_cols_numeric] = X[feature_cols_numeric].fillna(0)

print(f"Remaining NaNs in X: {X.isna().sum().sum()} (should be 0)")

# --- Random split (Week 3 style) ---
X_train_rand, X_test_rand, y_train_rand, y_test_rand = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y
)

# --- Grouped split, by client_hash_id ---
groups = df['client_hash_id']
gss = GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=42)
train_idx, test_idx = next(gss.split(X, y, groups=groups))
X_train_grp, X_test_grp = X.iloc[train_idx], X.iloc[test_idx]
y_train_grp, y_test_grp = y.iloc[train_idx], y.iloc[test_idx]

# Assert zero client overlap — the check the video insists on
train_clients = set(groups.iloc[train_idx])
test_clients = set(groups.iloc[test_idx])
overlap = train_clients & test_clients
print(f"Client overlap in grouped split: {len(overlap)} (should be 0)")
assert len(overlap) == 0, "Grouped split leaked a client across train/test"

print(f"Random split — train: {len(X_train_rand):,}, test: {len(X_test_rand):,}")
print(f"Grouped split — train: {len(X_train_grp):,}, test: {len(X_test_grp):,}, "
      f"train clients: {len(train_clients)}, test clients: {len(test_clients)}")

Remaining NaNs in X: 0 (should be 0)
Client overlap in grouped split: 0 (should be 0)
Random split — train: 22,137, test: 7,379
Grouped split — train: 14,175, test: 15,341, train clients: 22, test clients: 8


## 3. Train + compare vs my baseline

**Comparison contract:** every model below runs on the exact same feature matrix `X`, the exact
same two splits, and gets scored with the exact same metric — F1 on the `answered_away` class,
matching Week 3 and Week 4 so the number is comparable. The rule baseline itself doesn't get
retrained, it's Week 4's frozen `answered_away_high_position` / `answered_away_weak_position`
logic, scored on these same rows for a fair side-by-side.

The table below reports honest F1 for all three models, on both splits, so the reader can see
both "how good does this look" and "how much of that was the split doing the work."

In [20]:
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import f1_score
from sklearn.preprocessing import StandardScaler

results = []

def run_and_score(model, X_tr, X_te, y_tr, y_te, name, split_name, scale=False):
    if scale:
        scaler = StandardScaler()
        X_tr = scaler.fit_transform(X_tr)
        X_te = scaler.transform(X_te)
    model.fit(X_tr, y_tr)
    preds = model.predict(X_te)
    f1 = f1_score(y_te, preds)
    results.append({'model': name, 'split': split_name, 'f1_answered_away': round(f1, 3)})
    return model

# --- Random split ---
run_and_score(LogisticRegression(max_iter=1000, class_weight='balanced'),
              X_train_rand, X_test_rand, y_train_rand, y_test_rand,
              'Logistic Regression', 'random', scale=True)
run_and_score(DecisionTreeClassifier(max_depth=5, min_samples_leaf=50, random_state=42),
              X_train_rand, X_test_rand, y_train_rand, y_test_rand,
              'Decision Tree', 'random')
run_and_score(RandomForestClassifier(n_estimators=200, random_state=42),
              X_train_rand, X_test_rand, y_train_rand, y_test_rand,
              'Random Forest', 'random')

# --- Grouped split ---
run_and_score(LogisticRegression(max_iter=1000, class_weight='balanced'),
              X_train_grp, X_test_grp, y_train_grp, y_test_grp,
              'Logistic Regression', 'grouped', scale=True)
run_and_score(DecisionTreeClassifier(max_depth=5, min_samples_leaf=50, random_state=42),
              X_train_grp, X_test_grp, y_train_grp, y_test_grp,
              'Decision Tree', 'grouped')
run_and_score(RandomForestClassifier(n_estimators=200, random_state=42),
              X_train_grp, X_test_grp, y_train_grp, y_test_grp,
              'Random Forest', 'grouped')


results_df = pd.DataFrame(results)
print(results_df.to_string(index=False))

print("\n" + "="*70)
print("Note: I initially compared against Week 4's rule using pattern_group directly,")
print("but action=='structural_fix' IS defined as pattern_group=='answered_away' in")
print("Week 4's rule (position only changes the reason code, not the action) — so that")
print("comparison was circular (F1=1.000, comparing a column to itself), not a real test.")
print("Dropping it. The honest comparison is model F1 vs Week 3's honest F1=0.135, which")
print("used a Random Forest with no rule baseline in the loop at all.")
print("="*70)

print(f"\nWeek 3 honest F1 (reference, random split, no gsc_ctr_prior30 feature): 0.135")

              model   split  f1_answered_away
Logistic Regression  random             0.370
      Decision Tree  random             0.004
      Random Forest  random             0.149
Logistic Regression grouped             0.350
      Decision Tree grouped             0.014
      Random Forest grouped             0.108

Note: I initially compared against Week 4's rule using pattern_group directly,
but action=='structural_fix' IS defined as pattern_group=='answered_away' in
Week 4's rule (position only changes the reason code, not the action) — so that
comparison was circular (F1=1.000, comparing a column to itself), not a real test.
Dropping it. The honest comparison is model F1 vs Week 3's honest F1=0.135, which
used a Random Forest with no rule baseline in the loop at all.

Week 3 honest F1 (reference, random split, no gsc_ctr_prior30 feature): 0.135


## 4. Errors and interpretation

**What the numbers say:** Logistic Regression is the clear winner this week, beating Week 3's
honest F1 of 0.135 by a wide margin on both splits — 0.370 random, 0.350 grouped. The gap between
those two splits is small (0.020), which suggests this particular model isn't leaning heavily on
client-specific memorization to hit that score, unlike the framework video's own forest example
where the gap was large. That's a genuinely reassuring sign for this specific result.

Random Forest actually performed worse than Week 3's own Random Forest run (0.108 grouped vs
0.135 there). The false positive / false negative breakdown explains why: 453 false positives
against 3,516 false negatives on the grouped split, out of 15,341 test rows. The forest is
heavily under-flagging `answered_away` — it's being conservative almost across the board, not
missing a handful of edge cases. Looking at the ten sampled wrong cases, every one is a false
negative on a page with a low absolute click count (3-22 clicks) sitting outside page 1
(position 11-38), which matches the aggregate pattern: the model struggles most on pages where
the signal is real but small in absolute terms.

Decision Tree collapsed almost entirely (F1 of 0.004-0.014). With a 22.9% minority class and no
class weighting on the tree, `min_samples_leaf=50` most likely prevented the tree from ever
carving out a leaf that favors the minority class — it's predicting close to all-negative. This
is a real limitation of that specific configuration, not evidence the tree method itself can't
work here.

**Feature check:** the new feature earned its place. In the Random Forest, `gsc_ctr_prior30`
comes in at 0.227 importance, essentially tied with `gsc_avg_position_prior30` (0.234) and
`gsc_impressions_prior30` (0.229) as the three strongest signals — well ahead of `word_count`
(0.188) and `gsc_clicks_prior30` (0.101). In Logistic Regression, `word_count` and
`gsc_impressions_prior30` carry the largest weights (both negative), with `gsc_ctr_prior30`
contributing a smaller positive weight (0.058). The two models don't fully agree on which feature
matters most, which is itself worth noting rather than picking whichever story is more
convenient — it suggests the relationship isn't simply linear, since the tree-based model and the
linear model are drawing on the same signals differently.

**Why Random Forest underperforms Logistic Regression here:** Logistic Regression's
`class_weight='balanced'` directly reweights the loss function to

In [21]:
# Feature importance / coefficients — use the grouped-split models (the honest ones)
best_split_name = 'grouped'

log_reg_grp = LogisticRegression(max_iter=1000, class_weight='balanced')
scaler = StandardScaler()
X_train_grp_scaled = scaler.fit_transform(X_train_grp)
log_reg_grp.fit(X_train_grp_scaled, y_train_grp)

coef_df = pd.DataFrame({
    'feature': X.columns,
    'logreg_coef': log_reg_grp.coef_[0]
}).sort_values('logreg_coef', key=abs, ascending=False)
print("Logistic Regression coefficients (grouped split, sorted by |weight|):")
print(coef_df.head(10).to_string(index=False))

rf_grp = RandomForestClassifier(n_estimators=200, random_state=42)
rf_grp.fit(X_train_grp, y_train_grp)
importance_df = pd.DataFrame({
    'feature': X.columns,
    'rf_importance': rf_grp.feature_importances_
}).sort_values('rf_importance', ascending=False)
print("\nRandom Forest feature importances (grouped split):")
print(importance_df.head(10).to_string(index=False))

# Wrong cases from the Random Forest, grouped split
rf_preds_grp = rf_grp.predict(X_test_grp)
error_mask = rf_preds_grp != y_test_grp.values
error_rows = df.iloc[test_idx][error_mask][
    ['gsc_impressions_prior30', 'gsc_clicks_prior30', 'gsc_avg_position_prior30',
     'pattern_group']
].copy()
error_rows['predicted'] = rf_preds_grp[error_mask]
error_rows['actual'] = y_test_grp.values[error_mask]
print(f"\nTotal wrong cases: {error_mask.sum()} out of {len(y_test_grp)}")
print(error_rows.head(10).to_string(index=False))

fp_count = ((rf_preds_grp == 1) & (y_test_grp.values == 0)).sum()
fn_count = ((rf_preds_grp == 0) & (y_test_grp.values == 1)).sum()
print(f"\nFalse positives: {fp_count}")
print(f"False negatives: {fn_count}")

Logistic Regression coefficients (grouped split, sorted by |weight|):
                     feature  logreg_coef
                  word_count    -0.263359
     gsc_impressions_prior30    -0.170824
    gsc_avg_position_prior30     0.096086
 content_type_feedly article    -0.069043
content_type_keyword article     0.066287
             gsc_ctr_prior30     0.057806
   main_intent_informational    -0.047878
   main_intent_transactional     0.043616
             main_intent_nan     0.042489
    main_intent_navigational    -0.024457

Random Forest feature importances (grouped split):
                     feature  rf_importance
    gsc_avg_position_prior30       0.233907
     gsc_impressions_prior30       0.228868
             gsc_ctr_prior30       0.226966
                  word_count       0.187593
          gsc_clicks_prior30       0.101317
   main_intent_informational       0.006008
   main_intent_transactional       0.005619
      main_intent_commercial       0.005348
             main_in

## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.